# Lab: Representation Learning

The objectives of this lab is to build intuition on Gaussian Mixture Models as representations and as generative models; and how this connects to modern Genereative AI (via Variational Inference). As throughout the course, you should see an overlap between probabilistic models, and optimisation. 

We will start by using scikit-learn's implementation, but then at the end we will implement from scratch. Once you have completed the theoretical exercises, it should be clear what you need to implement for that. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal
from sklearn.mixture import GaussianMixture
from sklearn.datasets import fetch_openml

## Gaussian Mixture Models (GMM) and Expectation Maximization (EM)

We're going to use Gaussian Mixture Models for **density estimation**, i.e., describing $p(\mathbf{x})$, the distribution of the data.

Consider some data generated from Scikit-Learn's `make_moons` function: 

In [ ]:
from sklearn.datasets import make_moons

Xmoon, ymoon = make_moons(200, noise=.05, random_state=0)
plt.scatter(Xmoon[:, 0], Xmoon[:, 1]);
plt.xlabel('$x_1$');
plt.ylabel('$x_2$');

With the following code we can visually appreciate that 2 components are insufficient to model this data distribution: 

In [ ]:
def plot_gmm(gmm, X):
    z = gmm.fit(X).predict(X)

    # Plot contour plot
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.scatter(X[:, 0], X[:, 1], s=40, zorder=2)
    xl = ax.get_xlim()
    yl = ax.get_ylim()
    
    # Define grid for contour plot
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    # Calculate probabilities for each point on the grid
    zz = np.zeros_like(xx)
    for i in range(gmm.n_components):
        mean = gmm.means_[i][:2]  # Extract mean for the first 2 dimensions
        covariances = gmm.covariances_[i][:2, :2]  # Extract covariance matrix for the first 2 dimensions
        mvn = multivariate_normal(mean=mean, cov=covariances)
        zz += gmm.weights_[i] * mvn.pdf(np.stack([xx.ravel(), yy.ravel()]).T).reshape(xx.shape)

    plt.contourf(xx, yy, zz, alpha=0.5, levels=10)
    plt.title('$p(\\mathbf{x})$')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    ax.set_xlim(xl)
    ax.set_ylim(yl)

In [ ]:
gmm2 = GaussianMixture(n_components=2, covariance_type='full', random_state=0)
plot_gmm(gmm2, Xmoon)

But try using more components; sufficient to get a reasonable approximation of this distribution.

In [ ]:
### BEGIN SOLUTION
gmm_more_components = GaussianMixture(n_components=16, covariance_type='full', random_state=0)
### END SOLUTION
plot_gmm(gmm_more_components, Xmoon)

## Gaussian Mixture Models as a Generative Model

As well as serving as a probability density to evaluate $p(\mathbf{x})$, we can also use a GMM to _sample_ $\mathbf{x} \sim p(\mathbf{x})$, i.e., a **generative model**.

Use your `gmm_more_components` with the `sample` method to generate 400 new points. Place the result into `Xnew`. 

In [ ]:
# Xnew = ...
### BEGIN SOLUTION
Xnew, znew = gmm_more_components.sample(400)
### END SOLUTION
plt.scatter(Xnew[:, 0], Xnew[:, 1]);
plt.xlabel('$x_1$');
plt.ylabel('$x_2$');

## Choosing the number of components

What is the optimal number of components $m$?

We can simply evaluate the *likelihood* of the data under the model (as long as we take appropriate steps to avoid overfitting). In the following we make use of two criteria: [Akaike information criterion (AIC)](https://en.wikipedia.org/wiki/Akaike_information_criterion) and the [Bayesian information criterion (BIC)](https://en.wikipedia.org/wiki/Bayesian_information_criterion), to judge the quality of the mixture for a given number of components. Scikit-Learn's `GaussianMixture` estimator includes both, so there is not much code involved: 

In [ ]:
n_components_range = np.arange(1, 21)
models = [GaussianMixture(n, covariance_type='full', random_state=0).fit(Xmoon)
          for n in n_components_range]

plt.plot(n_components_range, [m.bic(Xmoon) for m in models], label='BIC')
plt.plot(n_components_range, [m.aic(Xmoon) for m in models], label='AIC')
plt.xlabel('$m$')
plt.xticks(n_components_range)
plt.legend(loc='best');

Which number of components do you choose? Note that the number of ideal components for modelling a distribution $p(\mathbf{x})$ is not necessarily the same number which is ideal for clustering $p(z=k | \mathbf{x})$.

In [ ]:
# n_components = 
### BEGIN SOLUTION
n_components = 6
### END SOLUTION

## Gaussian Mixture Models and PCA/Auto-Encoders

A major limitation of GMMs is that they are difficult to get good results from in high-dimensions. So if $\mathbf{x}$ is high dimensional (for example, the MNIST data), $p(\mathbf{x})$ will likely not be excellent. 

Let's address this issue by first obtaining a more compact representation, i.e., $\mathbf{z}$. You have already done exactly this kind of task in Lab 11 using linear and non-linear Auto-Encoders. 

First, let's get the MNIST data, and remind ourselves of what it looks like. 

Let's take only the 0s and 1s, to make the computation a little bit lighter. 

In [ ]:
# Fetch MNIST dataset
mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto')

# Extract features and labels
X, y = mnist['data'], mnist['target']
y = y.astype(int)

X = X[(y == 0) | (y == 1)]
y = y[(y == 0) | (y == 1)]

In [ ]:
X = np.array(X)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.flat):
    ax.imshow(X[i, :].reshape(28, 28), cmap='gray')
    ax.axis('off')

In [ ]:
# Use PCA (100 components) to create the dataset in `Z`-space:

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
# Z = ...
### BEGIN SOLUTION
pca = PCA(100, whiten=True)
Z = pca.fit_transform(X)
print(pca.n_components)
### END SOLUTION

Given this latent representation of the dataset, $\mathbf{z}_1,\ldots,\mathbf{z}_n$, try to estimate a the number of GMM components we should use (see the following figure):

**Hint** If this takes too long, start by using a random subsets of instances (several thousand - you may get warnings that the algorithm did not converge, so there is a tradeoff there), and `max_iter` with a reasonably small value (e.g., 5 or 10).

In [ ]:
# TODO some parameter search here

In [ ]:
# n_components = # enter your choice here
### BEGIN SOLUTION
n_components_range = np.arange(10, 200, 20)
models = [GaussianMixture(n, covariance_type='full', max_iter=5, random_state=0) for n in n_components_range]
Z_sampled = Z[np.random.choice(Z.shape[0], size=3000, replace=False), :]
aics = [model.fit(Z_sampled).aic(Z_sampled) for model in models]
plt.plot(n_components_range, aics);
plt.xlabel('n_components')
n_components = 100
### END SOLUTION

Let's remove the constraint on the number of iterations and check whether the algorithm has converged, on the full latent data `Z`: 

In [ ]:
gmm = GaussianMixture(n_components, covariance_type='full', random_state=0)
gmm.fit(Z)
print(gmm.converged_)

Draw 10 new points using your GMM as a generative model (put them in `Z_new`), and plot the results in data space (i.e., visualise the numbers, e.g., using `imshow` as in earlier tutorials). So, you'll need to reconstruct back into the data space, using the same instantiation of PCA as earlier (put them in `X_new`). 

In [ ]:
# Z_new =  # the drawn new samples in PC-space
# X_new =  # the drawn new samples in original data space
### BEGIN SOLUTION
Z_new, _ = gmm.sample(10)
X_new = pca.inverse_transform(Z_new)
### END SOLUTION
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_new[i, :].reshape(28, 28), cmap='gray')
    ax.axis('off')

## Towards Variational Auto Encoders (VAEs)

For this task, you should implement everything from scratch (EM for GMMs), but try to keep in mind the connection to VAEs.

A full 'deep' treatment (e.g., full VAE using PyTorch or similar) is out of scope for this lab; the emphasise should be on understanding the algorithmic solution and the connections to variational inference, which involve several important fundamentals of representation learning. 

In [ ]:
# Use the Xmoon data from above
X = Xmoon  

In [ ]:
# GMM parameters (initialization)
K = 30  # number of mixture components
n, d = X.shape
np.random.seed(0)

### BEGIN SOLUTION
pi = np.ones(K) / K
means = X[np.random.choice(n, K, replace=False)]
covs = [np.eye(d) for _ in range(K)]
### END SOLUTION

In [ ]:
# EM algorithm (variational view)
for t in range(20):
    # ----- E-step: update q(z_i = k) = p(z_i=k | x_i, theta)
    q = np.zeros((n, K))
    ### BEGIN SOLUTION
    for k in range(K):
        try:
            q[:, k] = pi[k] * multivariate_normal.pdf(X, mean=means[k], cov=covs[k])
        except:
            # Handle potential numerical issues with covariance matrices
            q[:, k] = pi[k] * multivariate_normal.pdf(X, mean=means[k], cov=covs[k] + 1e-6*np.eye(d))
    
    # Add small epsilon to avoid division by zero
    q = q + 1e-300
    q /= q.sum(axis=1, keepdims=True)  # normalize responsibilities
    ### END SOLUTION

    # ----- M-step: maximize ELBO wrt theta (pi, means, covs)
    ### BEGIN SOLUTION
    Nk = q.sum(axis=0)
    pi = Nk / n
    means = (q.T @ X) / Nk[:, None]
    
    for k in range(K):
        X_centered = X - means[k]
        weighted_cov = (q[:, k, None] * X_centered).T @ X_centered
        covs[k] = weighted_cov / Nk[k] + 1e-6*np.eye(d)
    ### END SOLUTION

In [ ]:
# Evaluate the mixture density p(x)
def p_x(x):
    # Return p_theta(x) for x (handles single point or array of points)
    ### BEGIN SOLUTION
    if x.ndim == 1:
        x = x.reshape(1, -1)
    density = np.zeros(x.shape[0])
    for k in range(K):
        try:
            density += pi[k] * multivariate_normal.pdf(x, mean=means[k], cov=covs[k])
        except:
            # Fallback for numerical issues
            density += pi[k] * multivariate_normal.pdf(x, mean=means[k], cov=covs[k] + 1e-6*np.eye(d))
    return density
    ### END SOLUTION

We will visualise the result.

In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Compute density on grid
zz = p_x(grid_points)
zz = zz.reshape(xx.shape)

# Compute density for data points
p = p_x(X)
p_scaled = p / p.max()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=p_scaled, cmap='viridis', s=30, alpha=0.7)
plt.contour(xx, yy, zz, levels=10, alpha=0.7, cmap='viridis')
#plt.colorbar(label=r'normalized $p(x)$')
plt.title(r"$p(x)$")
plt.axis('equal')

# Plot responsibilities
plt.subplot(1, 2, 2)
# Use the component with maximum responsibility for coloring
max_resp_component = np.argmax(q, axis=1)
plt.scatter(X[:, 0], X[:, 1], c=max_resp_component, cmap='tab20', s=30, alpha=0.7)
#plt.colorbar(label='Component index')
plt.title(r"$max q(z|x))$")
plt.axis('equal')

plt.tight_layout()
plt.show()

# Additional diagnostic plot: show the Gaussian components
plt.figure(figsize=(10, 8))
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.3, s=20)

for k in range(K):
    if pi[k] > 0.01:  # Only plot significant components
        # Draw ellipse representing covariance
        cov = covs[k]
        lambda_, v = np.linalg.eig(cov)
        lambda_ = np.sqrt(lambda_)
        
        ell = Ellipse(xy=means[k],
                      width=lambda_[0]*2, height=lambda_[1]*2,
                      angle=np.degrees(np.arctan2(v[1, 0], v[0, 0])),
                      alpha=0.3, color=f'C{k%10}')
        plt.gca().add_patch(ell)
        
        plt.scatter(means[k, 0], means[k, 1], marker='x', color='red', s=100)

plt.title("GMM Components (ellipses show 1 std)")
plt.axis('equal')
plt.show()

# Print some diagnostics
print(f"Component weights: {pi}")
print(f"Effective number of components: {(pi > 0.01).sum()}")